# Daily Merge

Merge raw session CSV files into one `merged_<animal>.csv` file per animal, then optionally merge all animals into `merged_all_subjects.csv` for the full cohort of the selected line.

## 1. Setup

Run this cell first. It makes imports work whether the notebook is launched from the repo root or from inside `notebooks/ASD` or `notebooks/Stakes`.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "analysis" / "daily_merge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

PosixPath('/Users/mafaldavalente/Documents/Mafalda_analysis')

## 2. Choose Dataset

Set `RAT = None` to process every animal in the cohort, or set it to one animal ID such as `"ASD0026"`.

In [2]:
LINE = "CNTNAP2"
COHORT = "cohort4"
RAT = None  # e.g. "ASD0026", or None for all animals

MODE = "both"  # "session", "animals", or "both"
MODEL_FILE = None

## 3. Optional Session Removals

Edit `SESSION_EDITS` before merging if a daily CSV should be removed entirely, or if only the bad tail/range of a session should be removed or marked repeated. File names must match the raw CSV names inside each animal folder.


In [ ]:
# Optional per-animal cleanup rules applied while creating merged_<animal>.csv.
# These affect the merged output only; they do not edit the raw daily CSV files.
#
# Actions:
# - drop_entire_session: skip that raw CSV completely
# - drop_from_trial: remove trials with trial >= start_trial
# - drop_trial_range: remove trials from start_trial through end_trial, inclusive
# - drop_block: remove one or more block values from a session file
# - mark_repeated_from: keep rows but set repeated_trial = True from start_trial onward

SESSION_EDITS = {
    # Previous provisions from DailyMerge.py.
    "ASD0013": [
        {"file": "out_ASD0013_251014.csv", "action": "mark_repeated_from", "start_trial": 6690},
    ],

    # Previous ASD0018 provisions from DailyMerge.py.
    # Change action to "drop_from_trial" if you want these rows removed instead.
    "ASD0018": [
        {"file": "ASD0018_out_251014.csv", "action": "mark_repeated_from", "start_trial": 7370},
        {"file": "ASD0018_out_251015.csv", "action": "mark_repeated_from", "start_trial": 8000},
        {"file": "out_ASD0018_251028.csv", "action": "mark_repeated_from", "start_trial": 10900},
        {"file": "out_ASD0018_251127.csv", "action": "mark_repeated_from", "start_trial": 22250},
    ],

    "ASD0052": [
        {"file": "out_ASD0052_260707.csv", "action": "mark_repeated_from", "start_trial": 39640},
    ],

    "ASD0058": [
        {"file": "out_ASD0058_260617.csv", "action": "drop_block", "blocks": [1, 2, 3]},
    ],

    "ASD0040": [
        {"file": "out_ASD0040_260715.csv", "action": "mark_repeated_from", "start_trial": 65140},
    ],

    # Examples for ASD0019. Uncomment/edit the raw filenames and thresholds as needed.
    # "ASD0019": [
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_entire_session"},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_from_trial", "start_trial": 5000},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_trial_range", "start_trial": 1000, "end_trial": 1500},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "block": 2},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_block", "blocks": [2, 3]},
    # ],
}

SESSION_EDITS


{'ASD0013': [{'file': 'out_ASD0013_251014.csv',
   'action': 'mark_repeated_from',
   'start_trial': 6690}],
 'ASD0018': [{'file': 'ASD0018_out_251014.csv',
   'action': 'mark_repeated_from',
   'start_trial': 7370},
  {'file': 'ASD0018_out_251015.csv',
   'action': 'mark_repeated_from',
   'start_trial': 8000},
  {'file': 'out_ASD0018_251028.csv',
   'action': 'mark_repeated_from',
   'start_trial': 10900},
  {'file': 'out_ASD0018_251127.csv',
   'action': 'mark_repeated_from',
   'start_trial': 22250}],
 'ASD0052': [{'file': 'out_ASD0052_260707.csv',
   'action': 'mark_repeated_from',
   'start_trial': 39640}],
 'ASD0058': [{'file': 'out_ASD0058_260617.csv',
   'action': 'drop_block',
   'blocks': [1, 2, 3]}]}

## 4. Optional Bad RT Values

Use `RT_VALUE_EDITS` when task outcomes and abort labels are valid, but the recorded numeric `timed_rt` values should be ignored in RT analyses. These edits keep the trials and only set `timed_rt` to missing in the merged outputs.


In [4]:
# Numeric RT values to ignore while keeping trials for accuracy/choice/abort analyses.
# This only blanks timed_rt and adds rt_value_valid / rt_value_note columns.
# It does not change success, abort_type, choices, trial counts, or repeated_trial.

RT_VALUE_EDITS = [
    {
        "setup": 2,
        "start_date": "2026-06-13",
        "end_date": "2026-06-18",  # update if the setup-2 issue continues
        "date_col": "source_date",
        "setup_col": "box",
        "rt_col": "timed_rt",
        "reason": "setup 2 RT value recording issue",
    },

]

RT_VALUE_EDITS


[{'setup': 2,
  'start_date': '2026-06-13',
  'end_date': '2026-06-18',
  'date_col': 'source_date',
  'setup_col': 'box',
  'rt_col': 'timed_rt',
  'reason': 'setup 2 RT value recording issue'}]

## 5. Preview Animals

Check which animals will be processed before writing merged files.

In [5]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import get_animals_for_cohort, get_base_dir

base_dir = get_base_dir(LINE, COHORT)
animals = get_animals_for_cohort(LINE, COHORT, rat=RAT)

print(f"Base directory: {base_dir}")
print(f"Animals ({len(animals)}): {animals}")

Base directory: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/CNTNAP2_cohort4
Animals (19): ['ASD0052', 'ASD0053', 'ASD0054', 'ASD0055', 'ASD0056', 'ASD0057', 'ASD0058', 'ASD0059', 'ASD0060', 'ASD0061', 'ASD0062', 'ASD0063', 'ASD0064', 'ASD0065', 'ASD0066', 'ASD0067', 'ASD0068', 'ASD0069', 'ASD0070']


## 6. Merge Daily Files Per Animal

This creates or updates `merged_<animal>.csv` files in the cohort data folder.

In [6]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import merge_session_files

if MODE in ("session", "both"):
    merge_session_files(
        line=LINE,
        cohort=COHORT,
        rat=RAT,
        session_edits=SESSION_EDITS,
        rt_value_edits=RT_VALUE_EDITS,
    )
else:
    print("Skipping per-animal session merge.")

Processing 19 animal(s) for CNTNAP2 cohort4: ASD0052, ASD0053, ASD0054, ASD0055, ASD0056, ASD0057, ASD0058, ASD0059, ASD0060, ASD0061, ASD0062, ASD0063, ASD0064, ASD0065, ASD0066, ASD0067, ASD0068, ASD0069, ASD0070
Using latest file 'out_ASD0052_260708.csv' as column reference (79 columns).
Total unique columns across all files: 92
✅ Added out_ASD0052_260420.csv (576 rows)
✅ Added out_ASD0052_260421.csv (938 rows)
✅ Added out_ASD0052_260422.csv (736 rows)
✅ Added out_ASD0052_260423.csv (791 rows)
✅ Added out_ASD0052_260424.csv (565 rows)
✅ Added out_ASD0052_260425.csv (556 rows)
✅ Added out_ASD0052_260427.csv (818 rows)
✅ Added out_ASD0052_260428.csv (675 rows)
✅ Added out_ASD0052_260429.csv (801 rows)
✅ Added out_ASD0052_260430.csv (850 rows)
✅ Added out_ASD0052_260501.csv (665 rows)
✅ Added out_ASD0052_260502.csv (678 rows)
✅ Added out_ASD0052_260504.csv (828 rows)
✅ Added out_ASD0052_260505.csv (741 rows)
✅ Added out_ASD0052_260507.csv (830 rows)
✅ Added out_ASD0052_260508.csv (936 

## 7. Merge Animals Into Cohort File

This creates or updates `merged_all_subjects.csv` in the cohort data folder.

In [7]:
import importlib
import analysis.daily_merge as DailyMerge
importlib.reload(DailyMerge)
from analysis.daily_merge import merge_subject_files_with_model

if MODE in ("animals", "both"):
    merged_df = merge_subject_files_with_model(
        line=LINE,
        cohort=COHORT,
        model_file=MODEL_FILE,
    )
else:
    merged_df = None
    print("Skipping cohort-level animal merge.")

if merged_df is not None:
    display(merged_df.head())
    print(merged_df.shape)

📘 Using model file 'merged_ASD0052.csv' with 98 columns.
🧾 Total unique columns across all subjects: 101


/Users/mafaldavalente/Documents/Mafalda_analysis/analysis/daily_merge.py:638: DtypeWarning: Columns (0: opto_onset) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Added merged_ASD0052.csv (40507 rows)


/Users/mafaldavalente/Documents/Mafalda_analysis/analysis/daily_merge.py:638: DtypeWarning: Columns (0: opto_onset, 1: rt_value_note) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Added merged_ASD0053.csv (28596 rows)


/Users/mafaldavalente/Documents/Mafalda_analysis/analysis/daily_merge.py:638: DtypeWarning: Columns (0: led0_pulses, 1: led1_pulses) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


✅ Added merged_ASD0054.csv (17918 rows)
✅ Added merged_ASD0055.csv (18837 rows)
✅ Added merged_ASD0056.csv (19759 rows)
✅ Added merged_ASD0057.csv (20564 rows)
✅ Added merged_ASD0058.csv (15697 rows)
✅ Added merged_ASD0059.csv (19842 rows)
✅ Added merged_ASD0060.csv (20913 rows)
✅ Added merged_ASD0061.csv (17513 rows)
✅ Added merged_ASD0062.csv (2232 rows)
✅ Added merged_ASD0063.csv (3398 rows)
✅ Added merged_ASD0064.csv (3002 rows)
✅ Added merged_ASD0065.csv (3522 rows)
✅ Added merged_ASD0066.csv (1427 rows)
✅ Added merged_ASD0067.csv (2078 rows)
✅ Added merged_ASD0068.csv (3760 rows)
✅ Added merged_ASD0069.csv (3074 rows)
✅ Added merged_ASD0070.csv (2465 rows)
🎉 Saved merged dataset: /Users/mafaldavalente/Documents/Mafalda_analysis/DataFiles/CNTNAP2_cohort4/merged_all_subjects.csv
Final shape: (245104, 101)


,animal,batch,experimenter,version,bias,repeated_trial,trial,trial_start,tared_trial_start,trial_end,...,lnp_start_frame,stim_dur,stim_dur_label,source_file,source_date,rt_value_valid,rt_value_note,trial_end_frame,lnp_end_frame,timed_ft_sot
0,ASD0052,cntnap2,MV,0.9.3,0.04,False,1,3.859519e+09,0.000000,3.859519e+09,...,4830.0,6000,RT,out_ASD0052_260420.csv,2026-04-20,True,NaN,NaN,NaN,NaN
1,ASD0052,cntnap2,MV,0.9.3,0.04,True,2,3.859519e+09,49.764992,3.859519e+09,...,NaN,6000,RT,out_ASD0052_260420.csv,2026-04-20,True,NaN,NaN,NaN,NaN
2,ASD0052,cntnap2,MV,0.9.3,0.08,False,3,3.859519e+09,231.792992,3.859519e+09,...,23569.0,6000,RT,out_ASD0052_260420.csv,2026-04-20,True,NaN,NaN,NaN,NaN
3,ASD0052,cntnap2,MV,0.9.3,0.08,True,4,3.859519e+09,237.388000,3.859519e+09,...,NaN,6000,RT,out_ASD0052_260420.csv,2026-04-20,True,NaN,NaN,NaN,NaN
4,ASD0052,cntnap2,MV,0.9.3,0.08,True,5,3.859519e+09,419.402976,3.859519e+09,...,NaN,6000,RT,out_ASD0052_260420.csv,2026-04-20,True,NaN,NaN,NaN,NaN


(245104, 101)
